In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path
import os
os.chdir(r"c:\Users\Huy\Documents\UIT\Sem_4\DS102")

FEATURES = 'data/final/features.csv'
TARGET = 'data/final/target.csv'
LABELS = [
    'SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON',
    'DELTA_LAT_6h', 'DELTA_LON_6h'
]

DROPS = ['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR'] + [
        col for col in [f'DELTA_LAT_{i}h' for i in [6, 12, 18, 24]]] + [
        col for col in [f'DELTA_LON_{i}h' for i in [6, 12, 18, 24]]]

XG_PARAMS = {
    'max_depth': 3,
    'learning_rate': 0.01,
    'n_estimators': 5000,
    'subsample': 0.8,
    'colsample_bytree': 0.7,
    'objective': 'reg:squarederror',
    'tree_method': 'hist', # Tăng tốc bằng histogram
    'random_state': 42,
    'n_jobs': -1           # Dùng toàn bộ nhân CPU
}

def haversine_distance(lat_true, lon_true, lat_pred, lon_pred):
    """Tính khoảng cách thực tế (km) trên mặt cầu Trái Đất"""
    lat_true, lon_true, lat_pred, lon_pred = map(np.radians, [lat_true, lon_true, lat_pred, lon_pred])
    dlat = lat_pred - lat_true
    dlon = lon_pred - lon_true
    a = np.sin(dlat/2.0)**2 + np.cos(lat_true) * np.cos(lat_pred) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c  # Bán kính Trái Đất ~ 6371 km

def prepare_data():
    features = pd.read_csv(FEATURES)
    target = pd.read_csv(TARGET)

    df = features.merge(target, on=['SID', 'YEAR', 'MONTH', 'DAY', 'HOUR', 'LAT', 'LON'], how='left')
    df = df.dropna(subset=['DELTA_LAT_6h', 'DELTA_LON_6h'])
    train_df = df[(df['YEAR'] >= 2000) & (df['YEAR'] <= 2020)]  # 21 years
    val_df = df[(df['YEAR'] >= 2021) & (df['YEAR'] <= 2022)]    # 2 years
    test_df = df[(df['YEAR'] >= 2023) & (df['YEAR'] <= 2024)]   # 2 years

    return train_df, val_df, test_df

In [2]:
train_df, val_df, test_df = prepare_data()
X_train = train_df.drop(columns=DROPS)
y_train_lat = train_df['DELTA_LAT_6h']
y_train_lon = train_df['DELTA_LON_6h']

X_val = val_df.drop(columns=DROPS)
y_val_lat = val_df['DELTA_LAT_6h']
y_val_lon = val_df['DELTA_LON_6h']

X_test = test_df.drop(columns=DROPS)
y_test_lat = test_df['DELTA_LAT_6h']
y_test_lon = test_df['DELTA_LON_6h']

current_lat_test = test_df['LAT'].values
current_lon_test = test_df['LON'].values

print("Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...")
model_lat = xgb.XGBRegressor(**XG_PARAMS, early_stopping_rounds=50)
model_lat.fit(
	X_train, y_train_lat, 
	eval_set=[(X_val, y_val_lat)], verbose=100
)

print("Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...")
model_lon = xgb.XGBRegressor(**XG_PARAMS, early_stopping_rounds=50)
model_lon.fit(
	X_train, y_train_lon,
	eval_set=[(X_val, y_val_lon)], verbose=100
)
pred_delta_lat = model_lat.predict(X_test)
pred_delta_lon = model_lon.predict(X_test)

pred_lat_6h = current_lat_test + pred_delta_lat
pred_lon_6h = current_lon_test + pred_delta_lon

true_lat_6h = current_lat_test + y_test_lat.values
true_lon_6h = current_lon_test + y_test_lon.values

test_errors_km = haversine_distance(true_lat_6h, true_lon_6h, pred_lat_6h, pred_lon_6h)
mean_error = np.mean(test_errors_km)

print(f"\n=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 6H TRUNG BÌNH (Mean Forecast Error): {mean_error:.2f} km")


Đang huấn luyện mô hình GBDT cho Vĩ độ (Latitude)...
[0]	validation_0-rmse:0.56485
[100]	validation_0-rmse:0.32744
[200]	validation_0-rmse:0.26857
[300]	validation_0-rmse:0.25314
[400]	validation_0-rmse:0.24659
[500]	validation_0-rmse:0.24380
[600]	validation_0-rmse:0.24237
[700]	validation_0-rmse:0.24150
[800]	validation_0-rmse:0.24074
[900]	validation_0-rmse:0.24033
[1000]	validation_0-rmse:0.24010
[1093]	validation_0-rmse:0.23989
Đang huấn luyện mô hình GBDT cho Kinh độ (Longitude)...
[0]	validation_0-rmse:0.95606
[100]	validation_0-rmse:0.45479
[200]	validation_0-rmse:0.32729
[300]	validation_0-rmse:0.30033
[400]	validation_0-rmse:0.29377
[500]	validation_0-rmse:0.29254
[534]	validation_0-rmse:0.29199

=> SAI SỐ KHOẢNG CÁCH DỰ BÁO 6H TRUNG BÌNH (Mean Forecast Error): 30.51 km
